Derm7pt -- Backbone Sweep -- LIGHTWEIGHT TIER ONLY (run on one of your 3 Kaggle accounts in parallel)

Before running:

Upload derm7pt_release_v0.zip (or the extracted release_v0 folder) as a private Kaggle Dataset on your main account, then add your other 2 accounts as collaborators on that dataset (Dataset page -> Settings -> Sharing -> Add collaborator).
On this account, attach the shared dataset to this notebook.
Notebook settings -> Accelerator: GPU T4 x2 (or whatever dual-GPU option you have). Internet: On (needed to download ImageNet-pretrained weights via timm).
Edit INPUT_DIR in the config cell below to match the attached dataset's path (check the Kaggle "Input" sidebar for the exact path).
This notebook only trains the lightweight tier. Run the other two tiers on your other 2 Kaggle accounts at the same time. After this finishes, merge sweep_results_lightweight.csv with the medium and heavy tier results (from your other 2 accounts) to build the full leaderboard.

In [ ]:
import os, time, csv, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import f1_score

try:
    import timm
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'])
    import timm

print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(' -', torch.cuda.get_device_name(i))

torch 2.11.0+cu128 | cuda available: True | GPUs: 2
 - Tesla T4
 - Tesla T4


In [ ]:
# ---- CONFIG ----
import glob

def _autodetect_input_dir():
    # works regardless of which account/dataset-slug mounted it, or how
    # deeply nested the release_v0 folder ends up under /kaggle/input
    hits = glob.glob('/kaggle/input/**/meta/meta.csv', recursive=True)
    if hits:
        return os.path.dirname(os.path.dirname(hits[0]))
    return None

_detected = _autodetect_input_dir()
if _detected:
    INPUT_DIR = _detected
    print('Auto-detected INPUT_DIR:', INPUT_DIR)
else:
    # fallback guesses -- known-good paths seen on this project's Kaggle accounts so far.
    # Edit/add to this list if auto-detect fails on a new account.
    _fallback_candidates = [
        '/kaggle/input/datasets/noorabdullah02/derm7pt-release-v0/release_v0',
        '/kaggle/input/derm7pt-release-v0/release_v0',
        '/kaggle/input/derm7pt/release_v0',
    ]
    INPUT_DIR = next((p for p in _fallback_candidates if os.path.isdir(p)), _fallback_candidates[0])
    print('Could not auto-detect dataset path (no meta/meta.csv found under /kaggle/input).')
    print('Falling back to', INPUT_DIR, '-- edit this manually if it is wrong.')

OUTPUT_DIR = '/kaggle/working'
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_PER_BACKBONE = 40  # early stopping below will cut this short once a backbone plateaus
EARLY_STOP_PATIENCE = 12
LR = 2e-4  # fine-tuning LR; gradient clipping + capped class weights keep this stable
NUM_WORKERS = 2

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MULTI_GPU = torch.cuda.device_count() > 1
os.makedirs(f'{OUTPUT_DIR}/checkpoints', exist_ok=True)

# If you attached the raw .zip instead of a pre-extracted folder, unzip it once:
if not os.path.isdir(INPUT_DIR):
    zip_candidates = [p for p in os.listdir('/kaggle/input') if p.endswith('.zip')]
    print('INPUT_DIR not found as a folder. Zip files seen in /kaggle/input:', zip_candidates)
    print('Either point INPUT_DIR at the extracted folder, or unzip manually into', OUTPUT_DIR)

Auto-detected INPUT_DIR: /kaggle/input/datasets/noorabdullah02/derm7pt-release-v0/release_v0


In [ ]:
# ---- labels.py ----
DIAGNOSIS_GROUPS = {
    'basal cell carcinoma': 'basal cell carcinoma', 'blue nevus': 'blue nevus',
    'clark nevus': 'clark nevus', 'combined nevus': 'combined nevus',
    'congenital nevus': 'congenital nevus', 'dermal nevus': 'dermal nevus',
    'dermatofibroma': 'dermatofibroma', 'lentigo': 'lentigo',
    'melanoma': 'melanoma', 'melanoma (in situ)': 'melanoma',
    'melanoma (less than 0.76 mm)': 'melanoma', 'melanoma (0.76 to 1.5 mm)': 'melanoma',
    'melanoma (more than 1.5 mm)': 'melanoma', 'melanoma metastasis': 'melanoma',
    'melanosis': 'melanosis', 'miscellaneous': 'miscellaneous',
    'recurrent nevus': 'recurrent nevus', 'reed or spitz nevus': 'reed or spitz nevus',
    'seborrheic keratosis': 'seborrheic keratosis', 'vascular lesion': 'vascular lesion',
}
DIAGNOSIS_CLASSES = sorted(set(DIAGNOSIS_GROUPS.values()))
MALIGNANT_DIAGNOSES = {'melanoma', 'basal cell carcinoma'}

SEVEN_POINT_CRITERIA = {
    'pigment_network': ['absent', 'typical', 'atypical'],
    'blue_whitish_veil': ['absent', 'present'],
    'vascular_structures': ['absent', 'arborizing', 'comma', 'hairpin', 'within regression', 'wreath', 'dotted', 'linear irregular'],
    'pigmentation': ['absent', 'diffuse regular', 'localized regular', 'diffuse irregular', 'localized irregular'],
    'streaks': ['absent', 'regular', 'irregular'],
    'dots_and_globules': ['absent', 'regular', 'irregular'],
    'regression_structures': ['absent', 'blue areas', 'white areas', 'combinations'],
}
CRITERIA_NUM_CLASSES = {k: len(v) for k, v in SEVEN_POINT_CRITERIA.items()}

def diagnosis_to_label(d):
    return DIAGNOSIS_GROUPS[d.strip().lower()]

def diagnosis_to_malignant(d):
    return int(diagnosis_to_label(d) in MALIGNANT_DIAGNOSES)

def criterion_to_index(col, value):
    return SEVEN_POINT_CRITERIA[col].index(value.strip().lower())

In [ ]:
# ---- dataset.py ----
CRITERIA_COLUMNS = list(SEVEN_POINT_CRITERIA.keys())

class Derm7ptDataset(Dataset):
    def __init__(self, release_dir, split, transform=None):
        assert split in ('train', 'valid', 'test')
        meta_dir = os.path.join(release_dir, 'meta')
        self.images_dir = os.path.join(release_dir, 'images')
        self.transform = transform
        meta = pd.read_csv(os.path.join(meta_dir, 'meta.csv'))
        split_idx = pd.read_csv(os.path.join(meta_dir, f'{split}_indexes.csv'))['indexes']
        self.df = meta.iloc[split_idx].reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        clinic_img = Image.open(os.path.join(self.images_dir, row['clinic'])).convert('RGB')
        derm_img = Image.open(os.path.join(self.images_dir, row['derm'])).convert('RGB')
        if self.transform is not None:
            clinic_img = self.transform(clinic_img)
            derm_img = self.transform(derm_img)
        diag_label = DIAGNOSIS_CLASSES.index(diagnosis_to_label(row['diagnosis']))
        malignant_label = diagnosis_to_malignant(row['diagnosis'])
        out = {
            'clinic': clinic_img, 'derm': derm_img,
            'diagnosis': torch.tensor(diag_label, dtype=torch.long),
            'malignant': torch.tensor(malignant_label, dtype=torch.long),
        }
        for col in CRITERIA_COLUMNS:
            out[col] = torch.tensor(criterion_to_index(col, row[col]), dtype=torch.long)
        return out

In [ ]:
# ---- model.py ----
class MultiTaskDerm7pt(nn.Module):
    def __init__(self, backbone_name, pretrained=True, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0, global_pool='avg')
        with torch.no_grad():
            feat_dim = self.backbone(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)).shape[1]
        fused_dim = feat_dim * 2
        self.dropout = nn.Dropout(dropout)
        self.diagnosis_head = nn.Linear(fused_dim, len(DIAGNOSIS_CLASSES))
        self.malignant_head = nn.Linear(fused_dim, 2)
        self.criteria_heads = nn.ModuleDict({n: nn.Linear(fused_dim, c) for n, c in CRITERIA_NUM_CLASSES.items()})
        task_names = ['diagnosis', 'malignant'] + list(CRITERIA_NUM_CLASSES.keys())
        self.log_vars = nn.ParameterDict({n: nn.Parameter(torch.zeros(())) for n in task_names})

    def forward(self, clinic, derm):
        fc = self.backbone(clinic)
        fd = self.backbone(derm)
        fused = self.dropout(torch.cat([fc, fd], dim=1))
        out = {'diagnosis': self.diagnosis_head(fused), 'malignant': self.malignant_head(fused)}
        for name, head in self.criteria_heads.items():
            out[name] = head(fused)
        return out

    def multitask_loss(self, outputs, targets):
        total = 0.0
        for name, log_var in self.log_vars.items():
            task_loss = F.cross_entropy(outputs[name], targets[name])
            total = total + torch.exp(-log_var) * task_loss + log_var
        return total

In [ ]:
# ---- engine.py ----
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
TASK_NAMES = ['diagnosis', 'malignant'] + list(CRITERIA_NUM_CLASSES.keys())

def build_transforms(img_size, train=True):
    if train:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)), transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15), transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
    return transforms.Compose([
        transforms.Resize((img_size, img_size)), transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

def expected_calibration_error(probs, labels, n_bins=10):
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    correct = (predictions == labels).astype(np.float64)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (confidences > lo) & (confidences <= hi)
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / len(labels)) * abs(correct[mask].mean() - confidences[mask].mean())
    return float(ece)

def train_one_epoch(model, raw_model, loader, optimizer):
    model.train()
    total_loss, n = 0.0, 0
    for batch in loader:
        clinic = batch['clinic'].to(DEVICE)
        derm = batch['derm'].to(DEVICE)
        targets = {k: batch[k].to(DEVICE) for k in TASK_NAMES}
        optimizer.zero_grad()
        outputs = model(clinic, derm)
        loss = raw_model.multitask_loss(outputs, targets)
        loss.backward()
        optimizer.step()
        bs = clinic.size(0)
        total_loss += loss.item() * bs
        n += bs
    return total_loss / max(n, 1)

@torch.no_grad()
def evaluate(model, raw_model, loader):
    model.eval()
    total_loss, n = 0.0, 0
    all_logits = {k: [] for k in TASK_NAMES}
    all_labels = {k: [] for k in TASK_NAMES}
    for batch in loader:
        clinic = batch['clinic'].to(DEVICE)
        derm = batch['derm'].to(DEVICE)
        targets = {k: batch[k].to(DEVICE) for k in TASK_NAMES}
        outputs = model(clinic, derm)
        loss = raw_model.multitask_loss(outputs, targets)
        bs = clinic.size(0)
        total_loss += loss.item() * bs
        n += bs
        for k in TASK_NAMES:
            all_logits[k].append(outputs[k].cpu())
            all_labels[k].append(targets[k].cpu())
    metrics = {'val_loss': total_loss / max(n, 1)}
    for k in TASK_NAMES:
        logits = torch.cat(all_logits[k]).numpy()
        labels = torch.cat(all_labels[k]).numpy()
        probs = F.softmax(torch.from_numpy(logits), dim=1).numpy()
        preds = probs.argmax(axis=1)
        metrics[f'{k}_acc'] = float((preds == labels).mean())
        metrics[f'{k}_macro_f1'] = float(f1_score(labels, preds, average='macro', zero_division=0))
        if k == 'diagnosis':
            metrics['diagnosis_ece'] = expected_calibration_error(probs, labels)
    return metrics

In [ ]:
# ---- backbone tier: THIS NOTEBOOK ONLY RUNS 'lightweight' ----
BACKBONE_TIERS = {
    'lightweight': ['mobilenetv3_small_100', 'mobilenetv3_large_100', 'efficientnet_b0', 'resnet18'],
}

In [ ]:
# ---- write standalone module files for the isolated-subprocess trainer ----
# (these are the same labels/dataset/model/engine logic as above, as real .py
# files on disk, plus the single-backbone trainer -- so each backbone can be
# trained in its own OS process via subprocess, immune to another backbone's
# GPU/CUDA crash.)
_MODULE_FILES = {
    "labels.py": '"""Label taxonomy for the Derm7pt dataset: diagnosis grouping + 7-point criteria encodings."""\n\n# Melanoma subtypes collapse into a single "melanoma" class (standard grouping,\n# matches the original JBHI 2019 paper\'s 15-way grouped diagnosis task).\nDIAGNOSIS_GROUPS = {\n    "basal cell carcinoma": "basal cell carcinoma",\n    "blue nevus": "blue nevus",\n    "clark nevus": "clark nevus",\n    "combined nevus": "combined nevus",\n    "congenital nevus": "congenital nevus",\n    "dermal nevus": "dermal nevus",\n    "dermatofibroma": "dermatofibroma",\n    "lentigo": "lentigo",\n    "melanoma": "melanoma",\n    "melanoma (in situ)": "melanoma",\n    "melanoma (less than 0.76 mm)": "melanoma",\n    "melanoma (0.76 to 1.5 mm)": "melanoma",\n    "melanoma (more than 1.5 mm)": "melanoma",\n    "melanoma metastasis": "melanoma",\n    "melanosis": "melanosis",\n    "miscellaneous": "miscellaneous",\n    "recurrent nevus": "recurrent nevus",\n    "reed or spitz nevus": "reed or spitz nevus",\n    "seborrheic keratosis": "seborrheic keratosis",\n    "vascular lesion": "vascular lesion",\n}\nDIAGNOSIS_CLASSES = sorted(set(DIAGNOSIS_GROUPS.values()))\n\n# Binary malignancy grouping, useful for a benign-vs-malignant head.\nMALIGNANT_DIAGNOSES = {"melanoma", "basal cell carcinoma"}\n\n# The seven 7-point-checklist criteria columns in meta.csv, with their label vocab.\nSEVEN_POINT_CRITERIA = {\n    "pigment_network": ["absent", "typical", "atypical"],\n    "blue_whitish_veil": ["absent", "present"],\n    "vascular_structures": [\n        "absent", "arborizing", "comma", "hairpin",\n        "within regression", "wreath", "dotted", "linear irregular",\n    ],\n    "pigmentation": [\n        "absent", "diffuse regular", "localized regular",\n        "diffuse irregular", "localized irregular",\n    ],\n    "streaks": ["absent", "regular", "irregular"],\n    "dots_and_globules": ["absent", "regular", "irregular"],\n    "regression_structures": ["absent", "blue areas", "white areas", "combinations"],\n}\n\n\ndef diagnosis_to_label(diagnosis: str) -> str:\n    return DIAGNOSIS_GROUPS[diagnosis.strip().lower()]\n\n\ndef diagnosis_to_malignant(diagnosis: str) -> int:\n    return int(diagnosis_to_label(diagnosis) in MALIGNANT_DIAGNOSES)\n\n\ndef criterion_to_index(column: str, value: str) -> int:\n    vocab = SEVEN_POINT_CRITERIA[column]\n    return vocab.index(value.strip().lower())\n',
    "dataset.py": '"""PyTorch Dataset for Derm7pt: loads clinical+dermoscopy image pairs with\ndiagnosis and 7-point-checklist criteria labels, using the official data splits.\n"""\nimport os\n\nimport pandas as pd\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import Dataset\n\nfrom labels import (\n    DIAGNOSIS_CLASSES,\n    SEVEN_POINT_CRITERIA,\n    criterion_to_index,\n    diagnosis_to_label,\n    diagnosis_to_malignant,\n)\n\nCRITERIA_COLUMNS = list(SEVEN_POINT_CRITERIA.keys())\n\n\nclass Derm7ptDataset(Dataset):\n    """One sample = one lesion case: a (clinical, dermoscopy) image pair,\n    the grouped diagnosis label, a binary malignancy label, and the 7\n    criteria labels (for multi-task training).\n    """\n\n    def __init__(self, release_dir: str, split: str, transform=None):\n        """\n        Args:\n            release_dir: path to .../derm7pt/release_v0\n            split: one of "train", "valid", "test"\n            transform: torchvision transform applied to both clinic & derm images\n        """\n        assert split in ("train", "valid", "test")\n        meta_dir = os.path.join(release_dir, "meta")\n        self.images_dir = os.path.join(release_dir, "images")\n        self.transform = transform\n\n        meta = pd.read_csv(os.path.join(meta_dir, "meta.csv"))\n        split_indexes = pd.read_csv(os.path.join(meta_dir, f"{split}_indexes.csv"))["indexes"]\n        self.df = meta.iloc[split_indexes].reset_index(drop=True)\n\n    def __len__(self):\n        return len(self.df)\n\n    def __getitem__(self, idx):\n        row = self.df.iloc[idx]\n\n        clinic_img = Image.open(os.path.join(self.images_dir, row["clinic"])).convert("RGB")\n        derm_img = Image.open(os.path.join(self.images_dir, row["derm"])).convert("RGB")\n        if self.transform is not None:\n            clinic_img = self.transform(clinic_img)\n            derm_img = self.transform(derm_img)\n\n        diag_label = DIAGNOSIS_CLASSES.index(diagnosis_to_label(row["diagnosis"]))\n        malignant_label = diagnosis_to_malignant(row["diagnosis"])\n        criteria_labels = {\n            col: criterion_to_index(col, row[col]) for col in CRITERIA_COLUMNS\n        }\n\n        return {\n            "clinic": clinic_img,\n            "derm": derm_img,\n            "diagnosis": torch.tensor(diag_label, dtype=torch.long),\n            "malignant": torch.tensor(malignant_label, dtype=torch.long),\n            **{\n                col: torch.tensor(val, dtype=torch.long)\n                for col, val in criteria_labels.items()\n            },\n            "case_num": int(row["case_num"]),\n        }\n\n\nif __name__ == "__main__":\n    import sys\n\n    release_dir = sys.argv[1] if len(sys.argv) > 1 else r"C:\\Users\\user\\Desktop\\DermPt7t\\derm7pt\\release_v0"\n    for split in ("train", "valid", "test"):\n        ds = Derm7ptDataset(release_dir, split)\n        print(f"{split}: {len(ds)} cases")\n        sample = ds[0]\n        print(\n            "  sample keys:", list(sample.keys()),\n            "| clinic size:", sample["clinic"].size,\n            "| diagnosis idx:", sample["diagnosis"].item(),\n        )\n',
    "model.py": '"""Multi-task, multi-modal (clinic + dermoscopy) model for Derm7pt.\n\nOne shared-weight image backbone (any timm model) encodes both the clinical\nand dermoscopic photo; the two feature vectors are concatenated and fed into\nper-task heads: diagnosis (15-way), malignant (binary), and the 7 checklist\ncriteria (each its own small classification head).\n"""\nimport timm\nimport torch\nimport torch.nn as nn\n\nfrom labels import DIAGNOSIS_CLASSES, SEVEN_POINT_CRITERIA\n\nCRITERIA_NUM_CLASSES = {k: len(v) for k, v in SEVEN_POINT_CRITERIA.items()}\n\n\nclass MultiTaskDerm7pt(nn.Module):\n    def __init__(self, backbone_name: str, pretrained: bool = True, dropout: float = 0.3):\n        super().__init__()\n        self.backbone = timm.create_model(\n            backbone_name, pretrained=pretrained, num_classes=0, global_pool="avg"\n        )\n        # `backbone.num_features` is unreliable for some archs (e.g. MobileNetV3\'s\n        # head-expansion conv isn\'t reflected in it) — probe the real output dim instead.\n        with torch.no_grad():\n            feat_dim = self.backbone(torch.zeros(1, 3, 224, 224)).shape[1]\n        fused_dim = feat_dim * 2  # clinic + derm concatenated\n\n        self.dropout = nn.Dropout(dropout)\n        self.diagnosis_head = nn.Linear(fused_dim, len(DIAGNOSIS_CLASSES))\n        self.malignant_head = nn.Linear(fused_dim, 2)\n        self.criteria_heads = nn.ModuleDict(\n            {name: nn.Linear(fused_dim, n) for name, n in CRITERIA_NUM_CLASSES.items()}\n        )\n\n        # Homoscedastic uncertainty weighting (Kendall et al., 2018): one learnable\n        # log-variance per task, so easy tasks don\'t drown out hard ones during\n        # multi-task training, and the learned variances double as calibration signal.\n        task_names = ["diagnosis", "malignant"] + list(CRITERIA_NUM_CLASSES.keys())\n        self.log_vars = nn.ParameterDict(\n            {name: nn.Parameter(torch.zeros(())) for name in task_names}\n        )\n\n    def forward(self, clinic: torch.Tensor, derm: torch.Tensor) -> dict:\n        feat_clinic = self.backbone(clinic)\n        feat_derm = self.backbone(derm)\n        fused = self.dropout(torch.cat([feat_clinic, feat_derm], dim=1))\n\n        out = {\n            "diagnosis": self.diagnosis_head(fused),\n            "malignant": self.malignant_head(fused),\n        }\n        for name, head in self.criteria_heads.items():\n            out[name] = head(fused)\n        return out\n\n    def multitask_loss(\n        self,\n        outputs: dict,\n        targets: dict,\n        label_smoothing: float = 0.1,\n        diagnosis_class_weights: torch.Tensor | None = None,\n    ) -> tuple[torch.Tensor, dict]:\n        """Uncertainty-weighted sum of per-task cross-entropy losses.\n        loss_i = exp(-log_var_i) * CE_i + log_var_i  (Kendall et al., 2018)\n\n        Label smoothing combats the overconfidence/poor calibration seen when\n        fine-tuning on this small a dataset (train loss -> 0 while val ECE rises).\n        diagnosis_class_weights (inverse-frequency) counters the severe class\n        imbalance in the 15-way diagnosis task (some classes have <30 cases).\n        """\n        ce = nn.functional.cross_entropy\n        per_task = {}\n        total = 0.0\n        for name, log_var in self.log_vars.items():\n            weight = diagnosis_class_weights if name == "diagnosis" else None\n            task_loss = ce(outputs[name], targets[name], weight=weight, label_smoothing=label_smoothing)\n            weighted = torch.exp(-log_var) * task_loss + log_var\n            per_task[name] = task_loss.detach()\n            total = total + weighted\n        return total, per_task\n\n\nif __name__ == "__main__":\n    import sys\n\n    backbone_name = sys.argv[1] if len(sys.argv) > 1 else "mobilenetv3_small_100"\n    model = MultiTaskDerm7pt(backbone_name, pretrained=False)\n    clinic = torch.randn(2, 3, 224, 224)\n    derm = torch.randn(2, 3, 224, 224)\n    out = model(clinic, derm)\n    print(f"backbone={backbone_name}  params={sum(p.numel() for p in model.parameters()):,}")\n    for k, v in out.items():\n        print(f"  {k}: {tuple(v.shape)}")\n',
    "engine.py": '"""Training / evaluation loops, metrics (incl. calibration), for the\nDerm7pt multi-task model. Shared by local CPU smoke tests and the Kaggle\nGPU sweep so results are produced the same way everywhere.\n"""\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom sklearn.metrics import f1_score\nfrom torchvision import transforms\n\nfrom model import CRITERIA_NUM_CLASSES\n\nIMAGENET_MEAN = [0.485, 0.456, 0.406]\nIMAGENET_STD = [0.229, 0.224, 0.225]\n\nTASK_NAMES = ["diagnosis", "malignant"] + list(CRITERIA_NUM_CLASSES.keys())\n\n\ndef build_transforms(img_size: int = 224, train: bool = True):\n    if train:\n        return transforms.Compose([\n            transforms.RandomResizedCrop(img_size, scale=(0.75, 1.0)),\n            transforms.RandomHorizontalFlip(),\n            transforms.RandomRotation(20),\n            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),\n            transforms.ToTensor(),\n            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),\n        ])\n    return transforms.Compose([\n        transforms.Resize((img_size, img_size)),\n        transforms.ToTensor(),\n        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),\n    ])\n\n\ndef compute_diagnosis_class_weights(dataset) -> torch.Tensor:\n    """Inverse-frequency weights for the 15-way diagnosis task, computed from\n    a (possibly Subset-wrapped) Derm7ptDataset\'s train split -- counters the\n    severe imbalance (e.g. dermatofibroma/vascular lesion < 30 cases each).\n    """\n    from model import DIAGNOSIS_CLASSES\n\n    counts = torch.zeros(len(DIAGNOSIS_CLASSES))\n    underlying = dataset.dataset if hasattr(dataset, "dataset") else dataset\n    indices = dataset.indices if hasattr(dataset, "indices") else range(len(dataset))\n    for i in indices:\n        counts[underlying[i]["diagnosis"].item()] += 1\n    weights = 1.0 / counts.clamp(min=1)\n    weights = weights / weights.mean()\n    # cap at 10x the average weight so one near-empty class (e.g. a single-digit\n    # case count) can\'t dominate the gradient and destabilize training\n    weights = weights.clamp(max=10.0)\n    return weights\n\n\ndef _targets_from_batch(batch: dict) -> dict:\n    return {name: batch[name] for name in TASK_NAMES}\n\n\ndef train_one_epoch(model, loader, optimizer, device, raw_model=None, diagnosis_class_weights=None) -> float:\n    loss_model = raw_model if raw_model is not None else model\n    model.train()\n    total_loss = 0.0\n    n = 0\n    for batch in loader:\n        clinic = batch["clinic"].to(device)\n        derm = batch["derm"].to(device)\n        targets = {k: v.to(device) for k, v in _targets_from_batch(batch).items()}\n\n        optimizer.zero_grad()\n        outputs = model(clinic, derm)\n        loss, _ = loss_model.multitask_loss(outputs, targets, diagnosis_class_weights=diagnosis_class_weights)\n        loss.backward()\n        # the rare-class weighting can occasionally produce a large gradient spike\n        # (e.g. a batch containing a near-single-digit-count class) -- clip it so\n        # one such batch can\'t destabilize the whole training run\n        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)\n        optimizer.step()\n\n        bs = clinic.size(0)\n        total_loss += loss.item() * bs\n        n += bs\n    return total_loss / max(n, 1)\n\n\ndef expected_calibration_error(probs: np.ndarray, labels: np.ndarray, n_bins: int = 10) -> float:\n    """ECE for the top predicted class: how far confidence is from actual accuracy."""\n    confidences = probs.max(axis=1)\n    predictions = probs.argmax(axis=1)\n    correct = (predictions == labels).astype(np.float64)\n\n    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)\n    ece = 0.0\n    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):\n        mask = (confidences > lo) & (confidences <= hi)\n        if mask.sum() == 0:\n            continue\n        bin_acc = correct[mask].mean()\n        bin_conf = confidences[mask].mean()\n        ece += (mask.sum() / len(labels)) * abs(bin_acc - bin_conf)\n    return float(ece)\n\n\n@torch.no_grad()\ndef evaluate(model, loader, device, raw_model=None) -> dict:\n    loss_model = raw_model if raw_model is not None else model\n    model.eval()\n    total_loss = 0.0\n    n = 0\n    all_logits = {name: [] for name in TASK_NAMES}\n    all_labels = {name: [] for name in TASK_NAMES}\n\n    for batch in loader:\n        clinic = batch["clinic"].to(device)\n        derm = batch["derm"].to(device)\n        targets = {k: v.to(device) for k, v in _targets_from_batch(batch).items()}\n\n        outputs = model(clinic, derm)\n        loss, _ = loss_model.multitask_loss(outputs, targets)\n        bs = clinic.size(0)\n        total_loss += loss.item() * bs\n        n += bs\n\n        for name in TASK_NAMES:\n            all_logits[name].append(outputs[name].cpu())\n            all_labels[name].append(targets[name].cpu())\n\n    metrics = {"val_loss": total_loss / max(n, 1)}\n    for name in TASK_NAMES:\n        logits = torch.cat(all_logits[name]).numpy()\n        labels = torch.cat(all_labels[name]).numpy()\n        probs = F.softmax(torch.from_numpy(logits), dim=1).numpy()\n        preds = probs.argmax(axis=1)\n\n        metrics[f"{name}_acc"] = float((preds == labels).mean())\n        metrics[f"{name}_macro_f1"] = float(\n            f1_score(labels, preds, average="macro", zero_division=0)\n        )\n        if name == "diagnosis":\n            metrics["diagnosis_ece"] = expected_calibration_error(probs, labels)\n\n    return metrics\n',
    "train_single_backbone.py": '"""Standalone single-backbone trainer for the Derm7pt multi-task sweep.\n\nRuns as its own OS process (invoked via subprocess.run from the Kaggle sweep\nnotebook) so a GPU/CUDA crash on one backbone (DataParallel + certain archs\nhas shown flaky cuDNN/misaligned-address failures on Kaggle\'s T4 pairs) can\nnever take down the rest of the sweep -- each backbone gets a fresh process\nand a fresh CUDA context.\n\nAlso bakes in the fixes for the accuracy/calibration problems seen in the\nfirst lightweight-tier run: label smoothing + diagnosis class weights (the\nmodel was overfitting fast -- train_loss -> ~0 while val ECE rose to 0.2-0.3\nand macro-F1 stayed far below accuracy because of severe class imbalance),\nstronger augmentation, a lower LR with cosine decay, and early stopping.\n"""\nimport argparse\nimport json\nimport os\nimport sys\nimport time\n\nimport torch\nimport torch.nn as nn\nfrom torch.utils.data import DataLoader\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\n\nfrom dataset import Derm7ptDataset\nfrom engine import build_transforms, compute_diagnosis_class_weights, evaluate, train_one_epoch\nfrom model import MultiTaskDerm7pt\n\n\ndef get_args():\n    p = argparse.ArgumentParser()\n    p.add_argument("--input-dir", required=True)\n    p.add_argument("--output-dir", required=True)\n    p.add_argument("--backbone", required=True)\n    p.add_argument("--tier", required=True)\n    p.add_argument("--epochs", type=int, default=40)\n    p.add_argument("--patience", type=int, default=12)\n    p.add_argument("--batch-size", type=int, default=32)\n    p.add_argument("--lr", type=float, default=2e-4)\n    p.add_argument("--img-size", type=int, default=224)\n    p.add_argument("--num-workers", type=int, default=2)\n    p.add_argument("--result-path", required=True)\n    p.add_argument("--limit-train", type=int, default=None, help="debug: cap train set size")\n    p.add_argument("--limit-valid", type=int, default=None, help="debug: cap valid set size")\n    p.add_argument("--no-pretrained", action="store_true", help="debug: skip downloading pretrained weights")\n    return p.parse_args()\n\n\ndef fallback_row(args, error, dt=None):\n    return {\n        "tier": args.tier, "backbone": args.backbone, "params": None,\n        "train_time_s": round(dt, 1) if dt else None, "val_loss": None,\n        "diagnosis_acc": None, "diagnosis_macro_f1": None, "diagnosis_ece": None,\n        "malignant_acc": None, "malignant_macro_f1": None, "error": str(error),\n    }\n\n\ndef main():\n    args = get_args()\n    t0 = time.time()\n    try:\n        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n        multi_gpu = torch.cuda.device_count() > 1\n\n        train_tf = build_transforms(args.img_size, train=True)\n        valid_tf = build_transforms(args.img_size, train=False)\n        train_ds = Derm7ptDataset(args.input_dir, "train", transform=train_tf)\n        valid_ds = Derm7ptDataset(args.input_dir, "valid", transform=valid_tf)\n        diagnosis_class_weights = compute_diagnosis_class_weights(train_ds).to(device)\n\n        if args.limit_train:\n            from torch.utils.data import Subset\n            train_ds = Subset(train_ds, range(min(args.limit_train, len(train_ds))))\n        if args.limit_valid:\n            from torch.utils.data import Subset\n            valid_ds = Subset(valid_ds, range(min(args.limit_valid, len(valid_ds))))\n\n        train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True,\n                                   num_workers=args.num_workers)\n        valid_loader = DataLoader(valid_ds, batch_size=args.batch_size, shuffle=False,\n                                   num_workers=args.num_workers)\n\n        raw_model = MultiTaskDerm7pt(args.backbone, pretrained=not args.no_pretrained).to(device)\n        model = nn.DataParallel(raw_model) if multi_gpu else raw_model\n        optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr)\n        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)\n\n        os.makedirs(f"{args.output_dir}/checkpoints", exist_ok=True)\n        best_acc, best_metrics, patience_ctr = -1.0, None, 0\n\n        for epoch in range(1, args.epochs + 1):\n            train_loss = train_one_epoch(model, train_loader, optimizer, device,\n                                          raw_model=raw_model,\n                                          diagnosis_class_weights=diagnosis_class_weights)\n            metrics = evaluate(model, valid_loader, device, raw_model=raw_model)\n            scheduler.step()\n\n            if metrics["diagnosis_acc"] > best_acc:\n                best_acc = metrics["diagnosis_acc"]\n                best_metrics = metrics\n                patience_ctr = 0\n                torch.save(raw_model.state_dict(),\n                           f"{args.output_dir}/checkpoints/{args.backbone}_best.pt")\n            else:\n                patience_ctr += 1\n\n            if epoch % 5 == 0 or epoch == args.epochs or patience_ctr == 0:\n                print(f"  epoch {epoch}/{args.epochs}  train_loss={train_loss:.4f}  "\n                      f"val_diag_acc={metrics[\'diagnosis_acc\']:.4f}  "\n                      f"val_diag_f1={metrics[\'diagnosis_macro_f1\']:.4f}  "\n                      f"ece={metrics[\'diagnosis_ece\']:.4f}  lr={scheduler.get_last_lr()[0]:.2e}")\n\n            if patience_ctr >= args.patience:\n                print(f"  early stopping at epoch {epoch} (no improvement for {args.patience} epochs)")\n                break\n\n        dt = time.time() - t0\n        n_params = sum(p.numel() for p in raw_model.parameters())\n        row = {"tier": args.tier, "backbone": args.backbone, "params": n_params,\n               "train_time_s": round(dt, 1), **best_metrics}\n        with open(args.result_path, "w") as f:\n            json.dump(row, f)\n        print(f"  -> best diagnosis_acc={best_acc:.4f}  ({dt:.1f}s, {n_params:,} params)")\n\n    except Exception as e:\n        print(f"  !! {args.backbone} FAILED: {e}")\n        with open(args.result_path, "w") as f:\n            json.dump(fallback_row(args, e, time.time() - t0), f)\n        sys.exit(1)\n\n\nif __name__ == "__main__":\n    main()\n',
}
for _fname, _content in _MODULE_FILES.items():
    with open(os.path.join(OUTPUT_DIR, _fname), "w", encoding="utf-8") as _f:
        _f.write(_content)
print("wrote subprocess trainer files to", OUTPUT_DIR, ":", list(_MODULE_FILES.keys()))

wrote subprocess trainer files to /kaggle/working : ['labels.py', 'dataset.py', 'model.py', 'engine.py', 'train_single_backbone.py']


In [ ]:
# ---- the sweep: train every backbone, every tier, each in its own subprocess ----
import subprocess
import sys
import json as _json

# sanity-check the data path up front (fails fast with a clear error, instead
# of only discovering a bad INPUT_DIR after launching the first subprocess)
_check_tf = build_transforms(IMG_SIZE, train=False)
_train_check = Derm7ptDataset(INPUT_DIR, 'train', transform=_check_tf)
_valid_check = Derm7ptDataset(INPUT_DIR, 'valid', transform=_check_tf)
print(f'train={len(_train_check)}  valid={len(_valid_check)}  (sanity check -- actual training runs in isolated subprocesses)')

RESULTS_DIR = f'{OUTPUT_DIR}/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

results = []
overall_best = {'diagnosis_acc': -1.0, 'backbone': None, 'tier': None}

for tier_name, backbones in BACKBONE_TIERS.items():
    print(f'\n========== TIER: {tier_name} ==========')
    tier_best_acc = -1.0
    for backbone_name in backbones:
        print(f'\n--- {backbone_name} (isolated subprocess) ---')
        result_path = f'{RESULTS_DIR}/{tier_name}_{backbone_name}.json'
        cmd = [sys.executable, f'{OUTPUT_DIR}/train_single_backbone.py',
               '--input-dir', INPUT_DIR, '--output-dir', OUTPUT_DIR,
               '--backbone', backbone_name, '--tier', tier_name,
               '--epochs', str(EPOCHS_PER_BACKBONE), '--patience', str(EARLY_STOP_PATIENCE),
               '--batch-size', str(BATCH_SIZE), '--lr', str(LR), '--img-size', str(IMG_SIZE),
               '--num-workers', str(NUM_WORKERS), '--result-path', result_path]
        proc = subprocess.run(cmd)
        if proc.returncode != 0:
            print(f'  !! subprocess exited with code {proc.returncode} for {backbone_name} (see traceback above)')

        if os.path.exists(result_path):
            with open(result_path) as f:
                row = _json.load(f)
        else:
            row = {'tier': tier_name, 'backbone': backbone_name, 'params': None,
                   'train_time_s': None, 'val_loss': None, 'diagnosis_acc': None,
                   'diagnosis_macro_f1': None, 'diagnosis_ece': None,
                   'malignant_acc': None, 'malignant_macro_f1': None,
                   'error': 'no result file written (subprocess crashed before saving)'}
        results.append(row)

        acc = row.get('diagnosis_acc')
        if acc is not None:
            tier_best_acc = max(tier_best_acc, acc)
            if acc > overall_best['diagnosis_acc']:
                overall_best = {'diagnosis_acc': acc, 'backbone': backbone_name, 'tier': tier_name}
        else:
            print(f'  -> FAILED: {row.get("error", "unknown error")}')

    print(f'Tier {tier_name} best: {tier_best_acc:.4f}')

train=413  valid=203  (sanity check -- actual training runs in isolated subprocesses)

========== TIER: lightweight ==========

--- mobilenetv3_small_100 (isolated subprocess) ---


  epoch 1/40  train_loss=11.2554  val_diag_acc=0.0591  val_diag_f1=0.0423  ece=0.0451  lr=2.00e-04
  epoch 2/40  train_loss=10.0373  val_diag_acc=0.1084  val_diag_f1=0.1031  ece=0.0264  lr=1.99e-04
  epoch 3/40  train_loss=9.4479  val_diag_acc=0.2463  val_diag_f1=0.1966  ece=0.1056  lr=1.97e-04
  epoch 4/40  train_loss=8.8292  val_diag_acc=0.2956  val_diag_f1=0.2625  ece=0.1254  lr=1.95e-04
  epoch 5/40  train_loss=8.3106  val_diag_acc=0.3498  val_diag_f1=0.2603  ece=0.1319  lr=1.92e-04
  epoch 6/40  train_loss=7.8783  val_diag_acc=0.3645  val_diag_f1=0.2879  ece=0.1147  lr=1.89e-04
  epoch 8/40  train_loss=7.2229  val_diag_acc=0.4089  val_diag_f1=0.3005  ece=0.1156  lr=1.81e-04
  epoch 10/40  train_loss=6.5649  val_diag_acc=0.3744  val_diag_f1=0.2844  ece=0.0836  lr=1.71e-04
  epoch 13/40  train_loss=6.208

In [ ]:
# ---- leaderboard ----
df = pd.DataFrame(results).sort_values('diagnosis_acc', ascending=False)
df.to_csv(f'{OUTPUT_DIR}/sweep_results_lightweight.csv', index=False)
print(f"\nBEST OVERALL: {overall_best['backbone']} ({overall_best['tier']} tier), "
      f"diagnosis_acc={overall_best['diagnosis_acc']:.4f}")
print(f"Checkpoint saved at: {OUTPUT_DIR}/checkpoints/{overall_best['backbone']}_best.pt")
df[['tier', 'backbone', 'params', 'diagnosis_acc', 'diagnosis_macro_f1', 'diagnosis_ece',
    'malignant_acc', 'train_time_s']]

BEST OVERALL: mobilenetv3_large_100 (lightweight tier), diagnosis_acc=0.5567
Checkpoint saved at: /kaggle/working/checkpoints/mobilenetv3_large_100_best.pt

	tier	backbone	params	diagnosis_acc	diagnosis_macro_f1	diagnosis_ece	malignant_acc	train_time_s
1	lightweight	mobilenetv3_large_100	4317286	0.556650	0.439959	0.260464	0.807882	240.7
2	lightweight	efficientnet_b0	4122802	0.541872	0.407129	0.242464	0.837438	353.7
0	lightweight	mobilenetv3_small_100	1610070	0.467980	0.345176	0.160914	0.743842	271.9
3	lightweight	resnet18	11222646	0.408867	0.272670	0.210624	0.788177	324.9